# UHI Modelling V2

I am attempting a revised workflow for modelling UHI.
First, the model is now going to include weather variables obtained from OpenMeteo. The data will then be aggregated with the Landsat data and used to build the model with the workflow below:  
1. Identify the world's climates from an authority (like the Trewartha climate classification).
2. Get substantial satellite (which will eventually include those those climate categories) and weather data on countries in those regions across the months (seasons) in a particular year.
3. Train a model on the data, implementing train-validation-test split, and predict UHI intensity on the test set.
4. Group the test examples by climate and compute the RMSE scores across the climates.
5. Cluster the UHI features to identify the types and categories of UHI.
6. Then I, human, assess the errors and prediction accuracies across each manufactured UHI cluster and how they vary across climates.

***"This project predicts urban heat island intensity using satellite and environmental data and evaluates how prediction reliability and errors vary across global climates and urban thermal types."***

## Testing aggregation with minimal data

To test the flow of data:  
1. From the various countries
2. In the various climates
3. Once a week from Jan 1 2025 to Dec 31 2025
4. From Earth Engine (Landsat) and OpenMeteo

I will be using as minimal data as possible. This will look like:  
1. Lagos, Ontario, Helsinki, and Tehran
2. Aw, Dc, Dcb, Bsk
3. Once a month Jan 1 2025 to Dec 31 2025
4. From Earth Engine and OpenMeteo

## Fetching from Open-Meteo

Again, the cities I want to sample are:
1. Lagos, Nigeria
2. Ontario, Canada
3. Tehran, Iran
4. Helsinki, Finland

The variables I am getting from Open-meteo are:
1. Cloud Cover (Low)
2. Air Temperature (2m)
3. Ralative Humidity
4. Precipitation
5. Wind Speed (10m)

In [1]:
import ee
from spectral import get_spectral
from weather import process_city_weather

ee.Authenticate()
ee.Initialize()

c:\Software Projects\Data Projects\uhi-modelling\uhenv\Lib\site-packages\geemap\conversion.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
cities = [
    {"code": "CAN", "name": "Ontario"},
    {"code": "IRN", "name": "Tehran"},
    {"code": "NGA", "name": "Lagos"},
	{"code": "FIN", "name": "Uusimaa"},
]

date_range = ("2025-01-01", "2025-12-31")

for city in cities:
    get_spectral(city['code'], city['name'], date_range)
    process_city_weather(city['code'], date_range)


Data has already been processed at data/CAN_spectral_features.csv.

Data has already been processed at data/CAN_Full_UHI_Data.csv.

Data has already been processed at data/IRN_spectral_features.csv.

Data has already been processed at data/IRN_Full_UHI_Data.csv.

Data has already been processed at data/NGA_spectral_features.csv.

Data has already been processed at data/NGA_Full_UHI_Data.csv.

Data has already been processed at data/FIN_spectral_features.csv.

Data has already been processed at data/FIN_Full_UHI_Data.csv.



Now that we have fetched our data, we will add the `city` column and concatenate all four datasets.

In [3]:
import pandas as pd

can_data = pd.read_csv("data/CAN_Full_UHI_Data.csv")
fin_data = pd.read_csv("data/FIN_Full_UHI_Data.csv")
irn_data = pd.read_csv("data/IRN_Full_UHI_Data.csv")
nga_data = pd.read_csv("data/NGA_Full_UHI_Data.csv")

In [4]:
uhi_data = pd.concat([can_data, fin_data, irn_data, nga_data], keys = ["Ontario", "Uusimaa", "Tehran", "Lagos"])

uhi_data = uhi_data.reset_index(level = 0).rename(columns = {"level_0": "city"})

In [5]:
print(uhi_data.head())
print(uhi_data.info())

      city                       date   humidity  precipitation  wind_speed  \
0  Ontario  2025-01-01 08:00:00+00:00  94.679740            0.3   10.009036   
1  Ontario  2025-01-01 14:00:00+00:00  88.242770            0.0   20.671806   
2  Ontario  2025-01-01 20:00:00+00:00  76.298164            0.1   25.172340   
3  Ontario  2025-02-01 08:00:00+00:00  76.699220            0.0   15.503006   
4  Ontario  2025-02-01 14:00:00+00:00  67.192100            0.0    6.248360   

   cloud_cover_low  air_temperature      Albedo   Elevation         LST  \
0            100.0            -0.25  19140.6732  300.671021  277.975567   
1             52.0            -0.90  19140.6732  300.671021  277.975567   
2            100.0            -0.80  19140.6732  300.671021  277.975567   
3              0.0           -17.75  19140.6732  300.671021  277.975567   
4              0.0           -21.85  19140.6732  300.671021  277.975567   

   LandCover     MNDWI      NDBI      NDVI      SAVI   latitude  longitude

Let's clean up the data:
1. Convert date to datetime
2. Convert city to category
3. Convert LandCover to category

In [6]:
uhi_data["date"] = pd.to_datetime(uhi_data["date"])

uhi_data.loc[uhi_data["date"].dt.hour.isin(range(6, 11)), "time"] = "morning"
uhi_data.loc[uhi_data["date"].dt.hour.isin(range(12, 15)), "time"] = "afternoon"
uhi_data.loc[uhi_data["date"].dt.hour.isin(range(16, 23)), "time"] = "evening"
    
print(uhi_data.head())

      city                      date   humidity  precipitation  wind_speed  \
0  Ontario 2025-01-01 08:00:00+00:00  94.679740            0.3   10.009036   
1  Ontario 2025-01-01 14:00:00+00:00  88.242770            0.0   20.671806   
2  Ontario 2025-01-01 20:00:00+00:00  76.298164            0.1   25.172340   
3  Ontario 2025-02-01 08:00:00+00:00  76.699220            0.0   15.503006   
4  Ontario 2025-02-01 14:00:00+00:00  67.192100            0.0    6.248360   

   cloud_cover_low  air_temperature      Albedo   Elevation         LST  \
0            100.0            -0.25  19140.6732  300.671021  277.975567   
1             52.0            -0.90  19140.6732  300.671021  277.975567   
2            100.0            -0.80  19140.6732  300.671021  277.975567   
3              0.0           -17.75  19140.6732  300.671021  277.975567   
4              0.0           -21.85  19140.6732  300.671021  277.975567   

   LandCover     MNDWI      NDBI      NDVI      SAVI   latitude  longitude  \
0 

Apparently, our Albedo values are unusbale. We didn't apply the scaling necessary to make the values go from 0 to 1.

In [9]:
unique_pixels = uhi_data[["latitude", "longitude", "date", "city"]].drop_duplicates().reset_index(drop=True)
unique_pixels["date_only"] = unique_pixels["date"].dt.date
unique_pixels.to_csv("data/unique_pixels.csv", index=False)

In [10]:
import ee
import geemap
import pandas as pd

ee.Initialize()

pixels = pd.read_csv("data/unique_pixels.csv")  # must include city column

results_by_date = []

for (city, date_only), group in pixels.groupby(["city", "date_only"]):
    end_date = (pd.Timestamp(date_only) + pd.Timedelta(days=16)).strftime("%Y-%m-%d")
    
    features = [
        ee.Feature(
            ee.Geometry.Point([row["longitude"], row["latitude"]]),
            {"latitude": row["latitude"], "longitude": row["longitude"], "date_only": date_only}
        )
        for _, row in group.iterrows()
    ]
    fc = ee.FeatureCollection(features)
    
    image = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(fc)  # constrain to this city's points
        .filterDate(date_only, end_date)
        .sort("CLOUD_COVER")
        .first()
        .select(["SR_B2", "SR_B4", "SR_B5", "SR_B6", "SR_B7"])
        .multiply(0.0000275).add(-0.2)
    )
    
    albedo = image.expression(
        "0.356*B2 + 0.130*B4 + 0.373*B5 + 0.085*B6 + 0.072*B7 - 0.0018",
        {
            "B2": image.select("SR_B2"),
            "B4": image.select("SR_B4"),
            "B5": image.select("SR_B5"),
            "B6": image.select("SR_B6"),
            "B7": image.select("SR_B7"),
        }
    ).rename("Albedo")
    
    sampled = albedo.sampleRegions(
        collection=fc,
        properties=["latitude", "longitude", "date_only"],
        scale=30
    )
    
    results_by_date.append(sampled)

all_results = ee.FeatureCollection(results_by_date).flatten()
geemap.ee_to_csv(all_results, filename="data/albedo_corrected.csv")

Image.select: Parameter 'input' is required and may not be null.
